# Project 7: CNN from Scratch — Convolution & Pooling

**Chapter:** 10 — Convolutional Neural Networks

**Goal:** Implement convolution and pooling operations from scratch. Build a minimal CNN that classifies small images. Understand why CNNs beat fully-connected nets on image tasks.

**What you'll build:**
1. 2D convolution operation
2. Max pooling
3. A simple Conv → ReLU → Pool → FC pipeline
4. Edge detection and feature visualization

**Rules:** No PyTorch/TF/sklearn. NumPy + matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Synthetic Image Data

We'll generate simple 8×8 grayscale images: horizontal lines vs vertical lines.

In [ ]:
def make_line_images(n=200, size=8, seed=42):
    """Generate 8x8 images with horizontal (label 0) or vertical (label 1) lines."""
    rng = np.random.default_rng(seed)
    images = []
    labels = []
    for _ in range(n):
        img = rng.normal(scale=0.1, size=(size, size))  # background noise
        if rng.random() < 0.5:
            # Horizontal line
            row = rng.integers(1, size-1)
            img[row, :] += 1.0
            labels.append(0)
        else:
            # Vertical line
            col = rng.integers(1, size-1)
            img[:, col] += 1.0
            labels.append(1)
        images.append(img)
    return np.array(images), np.array(labels)

X_img, y_img = make_line_images(n=400)
print(f"Images: {X_img.shape}, Labels: {y_img.shape}")

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_img[i], cmap='gray', vmin=-0.5, vmax=1.5)
    ax.set_title(f"{'Horiz' if y_img[i]==0 else 'Vert'}")
    ax.axis('off')
plt.suptitle('Sample Images'); plt.tight_layout(); plt.show()

## 2. Convolution & Pooling — Your Turn

### The math

**2D Convolution** (valid, no padding):
$$\text{out}[i, j] = \sum_{m}\sum_{n} \text{input}[i+m, j+n] \cdot \text{filter}[m, n]$$

**Max Pooling** (2×2, stride 2):
$$\text{out}[i, j] = \max(\text{input}[2i:2i+2,\; 2j:2j+2])$$

In [ ]:
def conv2d(image, kernel):
    """Apply a single 2D convolution filter to an image (valid padding).
    
    Args:
        image: (H, W) numpy array
        kernel: (kH, kW) numpy array
    Returns:
        output: (H-kH+1, W-kW+1) numpy array
    """
    ### YOUR CODE ###
    raise NotImplementedError("Slide the kernel across the image, compute dot product at each position")


def max_pool2d(image, pool_size=2):
    """Max pooling with given pool size and stride = pool_size.
    
    Args:
        image: (H, W) numpy array
    Returns:
        output: (H//pool_size, W//pool_size) numpy array
    """
    ### YOUR CODE ###
    raise NotImplementedError("Take the max of each pool_size x pool_size block")


def relu(x):
    return np.maximum(0, x)

### Sanity checks

In [ ]:
# Test conv2d
test_img = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)
test_kernel = np.array([[1, 0], [0, -1]], dtype=float)
result = conv2d(test_img, test_kernel)
assert result.shape == (2, 2), f"Expected (2,2), got {result.shape}"
assert result[0, 0] == 1*1 + 2*0 + 4*0 + 5*(-1), f"Wrong value: {result[0,0]}"
print(f"✓ conv2d correct: {result.tolist()}")

# Test max_pool
test_pool = np.array([[1, 3, 2, 4], [5, 6, 7, 8], [3, 2, 1, 0], [1, 2, 3, 4]], dtype=float)
pooled = max_pool2d(test_pool, pool_size=2)
assert pooled.shape == (2, 2)
assert pooled[0, 0] == 6 and pooled[0, 1] == 8
print(f"✓ max_pool correct: {pooled.tolist()}")

print("\n🎉 All checks passed!")

### Gradient checking for conv2d

Verify your convolution is correct by checking it against a naive nested-loop implementation. This is simpler than the full numerical gradient check in P05/P06 since conv2d has no trainable parameters — we're just checking the operation itself.

In [ ]:
def naive_conv2d(image, kernel):
    """Reference implementation using explicit nested loops."""
    kH, kW = kernel.shape
    oH = image.shape[0] - kH + 1
    oW = image.shape[1] - kW + 1
    out = np.zeros((oH, oW))
    for i in range(oH):
        for j in range(oW):
            for m in range(kH):
                for n in range(kW):
                    out[i, j] += image[i + m, j + n] * kernel[m, n]
    return out

rng_check = np.random.default_rng(99)

# Test 1: random 6x6 image, 3x3 kernel
img6 = rng_check.normal(size=(6, 6))
ker3 = rng_check.normal(size=(3, 3))
assert np.allclose(conv2d(img6, ker3), naive_conv2d(img6, ker3), atol=1e-10)
print("✓ conv2d matches reference on 6×6 image, 3×3 kernel")

# Test 2: 1x1 kernel (identity-scale)
ker1 = np.array([[2.0]])
assert np.allclose(conv2d(img6, ker1), naive_conv2d(img6, ker1), atol=1e-10)
print("✓ conv2d correct with 1×1 kernel")

# Test 3: kernel same size as image (single output value)
img3 = rng_check.normal(size=(3, 3))
ker_full = rng_check.normal(size=(3, 3))
result = conv2d(img3, ker_full)
assert result.shape == (1, 1), f"Expected (1,1), got {result.shape}"
assert np.allclose(result, naive_conv2d(img3, ker_full), atol=1e-10)
print("✓ conv2d correct when kernel covers entire image")

# Test 4: max_pool output shapes
pool_in4 = rng_check.normal(size=(4, 4))
assert max_pool2d(pool_in4, pool_size=2).shape == (2, 2)
print("✓ max_pool2d: 4×4 → 2×2 with pool_size=2")

pool_in6 = rng_check.normal(size=(6, 6))
assert max_pool2d(pool_in6, pool_size=3).shape == (2, 2)
print("✓ max_pool2d: 6×6 → 2×2 with pool_size=3")

print("\n🎉 All reference checks passed!")

## 3. Visualize: Edge Detection Filters

In [ ]:
# Classic edge detection kernels
horizontal_filter = np.array([[-1, -1, -1],
                               [ 0,  0,  0],
                               [ 1,  1,  1]], dtype=float)

vertical_filter = np.array([[-1, 0, 1],
                             [-1, 0, 1],
                             [-1, 0, 1]], dtype=float)

# Apply to sample images
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for col in range(4):
    img = X_img[col]
    h_edges = conv2d(img, horizontal_filter)
    v_edges = conv2d(img, vertical_filter)
    
    axes[0, col].imshow(img, cmap='gray'); axes[0, col].set_title(f"Original ({'H' if y_img[col]==0 else 'V'})")
    axes[1, col].imshow(h_edges, cmap='RdBu'); axes[1, col].set_title("Horiz filter")
    axes[2, col].imshow(v_edges, cmap='RdBu'); axes[2, col].set_title("Vert filter")
    for row in range(3): axes[row, col].axis('off')

plt.tight_layout(); plt.show()

## 4. Mini CNN Pipeline (Forward Only)

Build the forward pass of a minimal classifier: Conv → ReLU → Pool → Flatten → Logistic Regression.

In [ ]:
class SimpleCNN:
    def __init__(self, n_filters=4, kernel_size=3, img_size=8):
        self.n_filters = n_filters
        self.kernel_size = kernel_size
        # Random filters
        scale = np.sqrt(2.0 / (kernel_size * kernel_size))
        self.filters = np.random.randn(n_filters, kernel_size, kernel_size) * scale
        
        # After conv (valid): (img_size - kernel_size + 1)
        # After pool (2x2): // 2
        conv_out = img_size - kernel_size + 1
        pool_out = conv_out // 2
        flat_size = n_filters * pool_out * pool_out
        
        # FC layer for classification
        self.w_fc = np.random.randn(flat_size) * 0.01
        self.b_fc = 0.0
    
    def forward_one(self, image):
        """Forward pass for one image.
        
        Steps:
        1. Apply each filter with conv2d
        2. ReLU activation
        3. Max pool each feature map
        4. Flatten all feature maps into one vector
        5. Logistic regression: sigmoid(w · flat + b)
        
        Returns:
            probability (float), feature_maps (list of arrays)
        """
        ### YOUR CODE ###
        raise NotImplementedError("Conv → ReLU → Pool → Flatten → Sigmoid")
    
    def predict(self, images):
        probs = np.array([self.forward_one(img)[0] for img in images])
        return (probs >= 0.5).astype(int)
    
    def accuracy(self, images, labels):
        return np.mean(self.predict(images) == labels)

In [ ]:
cnn = SimpleCNN(n_filters=4, kernel_size=3)
prob, fmaps = cnn.forward_one(X_img[0])
print(f"Output probability: {prob:.4f}")
print(f"Number of feature maps: {len(fmaps)}")
print(f"Feature map shape: {fmaps[0].shape}")

# Visualize learned feature maps
fig, axes = plt.subplots(1, cnn.n_filters + 1, figsize=(16, 3))
axes[0].imshow(X_img[0], cmap='gray'); axes[0].set_title('Input')
for i, fm in enumerate(fmaps):
    axes[i+1].imshow(fm, cmap='RdBu'); axes[i+1].set_title(f'Filter {i+1}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Experiments

### A: Hand-crafted vs random filters
Replace the random filters with the horizontal/vertical edge detectors. Does accuracy improve?

In [ ]:
cnn_hand = SimpleCNN(n_filters=2, kernel_size=3)
cnn_hand.filters[0] = horizontal_filter
cnn_hand.filters[1] = vertical_filter

# The FC layer still needs training — but even random FC + good filters
# should give better-than-chance feature maps.
_, fmaps_h = cnn_hand.forward_one(X_img[0])  # horizontal line image
_, fmaps_v = cnn_hand.forward_one(X_img[1])  # try different images

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for row, (img_idx, fmaps) in enumerate([(0, fmaps_h), (1, fmaps_v)]):
    axes[row, 0].imshow(X_img[img_idx], cmap='gray')
    axes[row, 0].set_title(f"Input ({'H' if y_img[img_idx]==0 else 'V'})")
    axes[row, 1].imshow(fmaps[0], cmap='RdBu'); axes[row, 1].set_title('Horiz filter')
    axes[row, 2].imshow(fmaps[1], cmap='RdBu'); axes[row, 2].set_title('Vert filter')
for ax in axes.flat: ax.axis('off')
plt.tight_layout(); plt.show()

**Your observation:** *(Which filter responds more to which type of line?)*

...

### B: Translation invariance
Move a vertical line from column 2 to column 6. Does the CNN still recognize it?

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for i, col in enumerate([1, 2, 3, 5, 6]):
    img = np.random.randn(8, 8) * 0.1
    img[:, col] += 1.0
    prob, fmaps = cnn_hand.forward_one(img)
    axes[0, i].imshow(img, cmap='gray'); axes[0, i].set_title(f'Line at col {col}')
    axes[1, i].imshow(fmaps[1], cmap='RdBu'); axes[1, i].set_title(f'Vert response')
for ax in axes.flat: ax.axis('off')
plt.suptitle('Translation Invariance: Same filter responds regardless of position')
plt.tight_layout(); plt.show()

## 6. What You Learned

1. What does a convolution filter detect? Why is it translation-invariant?
2. What does pooling do? What information does it preserve/discard?
3. Why are CNNs better than fully-connected nets for images?
4. In a real CNN, the filters are *learned* — why is that better than hand-crafting them?

---
**Next project:** Capstone — put it all together on a real dataset.